# N095 · 收支平衡与双向约束

**目标：** 用前缀亏损和两侧约束推导贪心。

**先修：** N031, N093。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 前缀最小；候选起点排除；两遍扫描；局部约束合并。

**配套讲解来源：** [同名逐章意见](../../comment/095_greedy_balance_two_pass.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这章把两道"表面不同、内核相似"的题放在一起：它们都需要**先分析约束的方向，再决定扫描的方式**。

**问题一（加油站，LeetCode 134）**：环形路上有 n 个加油站，第 i 个站能加油 `gas[i]` 升，从第 i 站开到下一站要耗 `cost[i]` 升。油箱无限大，问：从哪个站出发才能绕一圈回到原地？题目保证答案唯一；不存在则返回 -1。给你具体数字：`gas=[1,2,3,4,5]`、`cost=[3,4,5,1,2]`。每站的"净收入"是 gas−cost = [−2,−2,−2,3,3]。从 0 号出发：带 1 升上路，扣 3 直接亏空，不行；从 3 号出发：+3、+3、再绕回来补 −2、−2、−2，油箱依次是 3→6→4→2→0，全程没断油，**所以答案是 3**。暴力做法要每个起点都试一圈，是 O(n²)；本章一遍扫描就出结果。

**问题二（分糖果，LeetCode 135）**：一排孩子各有评分 `ratings[i]`，你要发糖果满足：(a) 每人至少 1 颗；(b) 相邻两个孩子里评分更高的那个必须拿更多（评分相同没有要求）。问最少发多少颗。具体数字（“运行示例”部分 实例）：`ratings=[1,3,4,5,2]`，答案是 **11 颗**——分配从左到右是 [1,2,3,4,1]：前四人评分递增各多拿一颗，最后那个评分 2 的小孩拿 1 颗（他只比左邻低，不比谁高）。为什么不能更少？因为"1<3<4<5"这条递增链逼着这四个位置至少是 1,2,3,4 颗，最后位置至少 1 颗，11 颗就是下界。

## 2. 基础知识：先读懂这些词

- **净收入 / 差值序列（gas − cost）**：加油站问题的第一变换。每个站对油箱的贡献就是"加的油减去走的油"，之后只看这条差值序列的累计和。
- **前缀亏损（prefix deficit）**：从某个候选起点累计到位置 i 时油箱转负，就叫"到 i 为止出现前缀亏损"。亏损一旦发生，当前候选就被判死刑，考察下一个候选。
- **候选起点排除（candidate elimination）**：核心引理——如果从起点 s 出发在位置 i 断油，那么 s+1、s+2、…、i 这些中间位置全都不用试了。理由是：从 s 一路开到 i 之前油箱都没断过（中途任意前缀和非负），中途任何位置 s' 的"家底"不会比从 s 带过来的更多，从 s' 出发到 i 的累计只会更少，同样断油。一次失败排除一整段，这是能把 O(n²) 压成 O(n) 的原因。
- **两遍扫描（two-pass）**：糖果题的解法框架。第一遍从左到右处理"比左邻居高"的约束，第二遍从右到左处理"比右邻居高"的约束，最后把两个方向的要求合并。
- **局部约束合并（pointwise max merge）**：每个位置有两套要求——左链要求它至少 L 颗、右链要求它至少 R 颗——两套都要满足就是取 `max(L, R)`。逐点取最大之后再无冗余，所以同时是"可行"且"最小"的。
- **必要下界（necessary lower bound）**：单看左侧约束，`left[i]` 是位置 i 能拿到的最少糖果（沿着左边递增链一路 +1 推出来的）；任何可行方案在这个位置都至少要发 `left[i]` 颗——这就是"下界"的含义。右侧同理。

## 3. 思路推导：从小例子开始

**加油站：一遍扫描**

1. **Step 1：** 转成差值思维：一路累计油箱 `tank += gas[i] - cost[i]`，同时累计总和 `total`。
2. **Step 2：** 一旦 `tank < 0`，说明当前候选起点失败；根据"排除引理"，直接把候选起点挪到 `i+1`，油箱清零重新累计。
3. **Step 3：** 扫完后看 `total`：总和都凑不齐 0（`total < 0`），说明全路净亏损，谁出发都白搭，返回 -1；否则最后留下的候选就是答案。

手算（`gas=[1,2,3,4,5]`、`cost=[3,4,5,1,2]`，差值 [−2,−2,−2,3,3]）逐站列表：

| i | 差值 g−c | tank（清零前） | 动作 |
|---|---|---|---|
| 0 | −2 | −2 | 断油，候选挪到 1，tank 清零 |
| 1 | −2 | −2 | 断油，候选挪到 2，tank 清零 |
| 2 | −2 | −2 | 断油，候选挪到 3，tank 清零 |
| 3 | +3 | 3 | 不断油，继续 |
| 4 | +3 | 6 | 不断油，扫完；total=0 ≥ 0，返回候选 3 |

三次失败一口气排除了起点 0、1、2，这就是排除引理的威力。

**糖果：两遍扫描 + 逐点取最大**

1. **Step 1（左遍）：** `left` 数组全 1；从 i=1 往右扫，如果 `ratings[i] > ratings[i-1]`，那么 `left[i] = left[i-1] + 1`。这一遍保证所有"比左邻高"的约束以最小代价满足。
2. **Step 2（右遍）：** 从最右往左扫，维护一个滚动变量 `right`（含义同 left，只是方向反了）：如果 `ratings[i] > ratings[i+1]`，`right` 在上一轮基础上 +1，否则重置为 1。
3. **Step 3（合并）：** 每个位置的最少糖果是 `max(left[i], right[i])`，边扫边累加。

手算（“运行示例”部分 的 `ratings=[1,3,4,5,2]`，“运行示例”部分 为了画表把 right 也存成了数组）：

| 评分 | 左约束 | 右约束 | 分配 = max |
|---|---|---|---|
| 1 | 1 | 1 | 1 |
| 3 | 2 | 1 | 2 |
| 4 | 3 | 1 | 3 |
| 5 | 4 | 2 | 4 |
| 2 | 1 | 1 | 1 |

左遍：评分 1<3<4<5 一路递增，left = [1,2,3,4]，最后评分 2 比左邻低，left[4]=1。右遍：只有评分 5 比右邻（评分 2）高，所以 right[3]=2，其余位置都是 1。逐点取 max 得 [1,2,3,4,1]，总计 11——这张表就是 “运行示例”部分 运行后打印的内容。注意评分 5 这个位置：左链要求它 ≥4、右链要求它 ≥2，两边都抬它，取大者 4。

## 4. 核心代码：can_complete_circuit

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def can_complete_circuit(gas, cost):
    if len(gas) != len(cost) or not gas:
        return -1
    total = tank = start = 0
    for i, (g, c) in enumerate(zip(gas, cost)):
        total += g - c
        tank += g - c
        if tank < 0:
            start = i + 1
            tank = 0
    return start if total >= 0 else -1
```

### 逐段读懂代码

**函数一：加油站（以下是 “分段实现”部分 的逐字代码）**

```python
def can_complete_circuit(gas, cost):
    if len(gas) != len(cost) or not gas:
        return -1
    total = tank = start = 0
    for i, (g, c) in enumerate(zip(gas, cost)):
        total += g - c
        tank += g - c
        if tank < 0:
            start = i + 1
            tank = 0
    return start if total >= 0 else -1
```

- 第 2 行在挡两类非法/边界输入：两个数组长度不一致，或数组为空，都直接返回 -1。这是本章显式写出来的防御。
- `total` 是全程净收支，`tank` 是从当前候选出发的油箱，`start` 是候选起点，三个都从 0 开始。
- `total+=g-c; tank+=g-c`：两个累计器同步记账——一个为最终判定（总收支是否非负），一个为中途判死（当前候选是否断油）。
- `if tank<0: start=i+1; tank=0`：断油处理。候选直接跳到 `i+1` 而不是 `start+1`，因为排除引理保证 s..i 这一段整段都是废案，跳一格一格试是浪费；`tank=0` 意思是"假装从新候选空箱出发重新累计"。
- `return start if total>=0 else -1`：最后的一道闸——扫描只保证了"从 start 到末尾没断油"，还要总和非负才保证绕回开头的路段也不断油（原因见第五节）。

**函数二：糖果（以下是 “分段实现”部分 的逐字代码）**

```python
def candy(ratings):
    n = len(ratings)
    left = [1] * n
    right = 1
    total = 0
    for i in range(1, n):
        if ratings[i] > ratings[i - 1]:
            left[i] = left[i - 1] + 1
    for i in range(n - 1, -1, -1):
        right = right + 1 if i < n - 1 and ratings[i] > ratings[i + 1] else 1
        total += max(left[i], right)
    return total
```

- `left=[1]*n`：左约束数组，先假设人人都拿最小值 1 颗（"每人至少 1"这条约束的落码）。
- 第一遍循环只处理向右递增：`ratings[i]>ratings[i-1]` 时左邻拿几颗我就多拿一颗。相等或更低都不动，保持 1——因为相邻同分不要求糖果数有差别，这正是"最小"的来源之一。
- 第二遍循环从右往左（`range(n-1,-1,-1)`）。`right` 是个滚动变量而不是数组：`right=right+1 if i<n-1 and ratings[i]>ratings[i+1] else 1` 这行是个条件表达式——当 i 还不是最右端且比右邻评分高时，`right` 接着上一轮（即右邻的值）加 1；否则重置成 1。两个条件缺一不可：`i<n-1` 是防止越界访问 `ratings[i+1]`（最右端没有右邻），`ratings[i]>ratings[i+1]` 是"比右邻高才加码"的题意。因为它每一轮要么 +1 要么归 1，滚动存它就够了，省一个 O(n) 数组。
- `total+=max(left[i],right)`：把左右两个下界当场合并累加，第二遍扫完总数就出来了，不需要再开一趟合并循环。

补一份"滚动 right 变量"的手算（用 “运行示例”部分 的 `ratings=[1,3,4,5,2]`），你可以对着上面的条件表达式逐轮核对：

| 轮次 i | 比较 ratings[i]>ratings[i+1] | right 取值 | left[i] | max 计入 |
|---|---|---|---|---|
| 4 | 不比较（i=n−1） | 1 | 1 | 1 |
| 3 | 5>2 成立 | 1+1=2 | 4 | 4 |
| 2 | 4>5 不成立 | 1 | 3 | 3 |
| 1 | 3>4 不成立 | 1 | 2 | 2 |
| 0 | 1>3 不成立 | 1 | 1 | 1 |

五行 max 加起来正好是 11，与第三节表格的分配 [1,2,3,4,1] 完全一致——只不过 “运行示例”部分 的演示把 right 存成了数组，而正式代码里它是一个每轮要么 +1、要么归 1 的滚动值，行为一模一样但省掉了 O(n) 空间。

## 5. 分段实现：按顺序运行

**本章接口：** `can_complete_circuit(gas, cost)`、`candy(ratings)`

### can_complete_circuit

In [1]:
def can_complete_circuit(gas, cost):
    if len(gas) != len(cost) or not gas:
        return -1
    total = tank = start = 0
    for i, (g, c) in enumerate(zip(gas, cost)):
        total += g - c
        tank += g - c
        if tank < 0:
            start = i + 1
            tank = 0
    return start if total >= 0 else -1

### candy

In [2]:
def candy(ratings):
    n = len(ratings)
    left = [1] * n
    right = 1
    total = 0
    for i in range(1, n):
        if ratings[i] > ratings[i - 1]:
            left[i] = left[i - 1] + 1
    for i in range(n - 1, -1, -1):
        right = right + 1 if i < n - 1 and ratings[i] > ratings[i + 1] else 1
        total += max(left[i], right)
    return total

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

ratings=[1,3,4,5,2]; left=[1]*len(ratings); right=[1]*len(ratings)
for i in range(1,len(ratings)):
    if ratings[i]>ratings[i-1]: left[i]=left[i-1]+1
for i in range(len(ratings)-2,-1,-1):
    if ratings[i]>ratings[i+1]: right[i]=right[i+1]+1
show_table(['评分','左约束','右约束','分配'],[(r,a,b,max(a,b)) for r,a,b in zip(ratings,left,right)])

评分,左约束,右约束,分配
1,1,1,1
3,2,1,2
4,3,1,3
5,4,2,4
2,1,1,1


## 7. 正确性、适用条件与复杂度

加油站扫描不变量：当前候选以前的起点已被一个负和区间排除；总和非负时，最后剩余候选可完成一圈。糖果左右数组分别是必要下界，其逐点最大同时满足两侧不等式，故不仅可行，而且逐点最小。

**代价：** 均O(n)时间；加油站O(1)辅助，糖果O(n)辅助。相邻同分不要求相同糖果；空评分总量0。

### 展开理解

**加油站为什么对？** 分两句说。

- 第一句（扫描中）：每次 `tank<0` 都发生在"累计差值的新低点"——从上一个候选出发到失败点之前油箱一直是非负的，所以失败点那一刻的累计值比这期间任何时刻都低；换句话说，每个被淘汰的候选都是被一段"净和为负"的具体路段杀死的，谁也喊不了冤。
- 第二句（绕圈时）：设最后活下来的候选是 s。从 s 扫到末尾油箱始终非负，设终点剩 R 升；注意 R = 总和 −（开头段的累计亏损），而失败点 s−1 恰好是全程累计值的最低谷，所以绕回开头后任何时刻的油量 = R + 开头段到那一点的累计值 ≥ R + 最低谷之后的回补 ≥ 总和 ≥ 0。也就是说"总和 ≥ 0"恰好兜住了绕圈的后半程，这就是最后一行 `if total>=0` 那道闸的底气。反过来，若总和 < 0，任何起点的油量终值都是负的，必然断油，返回 -1 合理。

**糖果为什么对？** 三个事实摞起来。

- 事实一：只看"比左邻高"这一组约束，`left[i]` 是位置 i 的最小取值——你可以沿着递增链一路推：链上第一个位置至少 1 颗，后一个比前一个评分高就必须至少多 1 颗，推到头别无选择。
- 事实二：同理，`right[i]` 是只看右侧约束时的最小取值。
- 事实三：一个方案要同时满足两组约束，每个位置就得同时不低于两个下界，即 `max(left[i], right[i])`；而取了逐点最大之后，两侧约束又确实都被满足（max 不小于任何一侧的要求）。所以逐点最大既是可行解，又每个位置都贴着下界，加起来就是最小总量。这也解释了为什么必须两遍扫描：约束是从左右两个方向压过来的，只从一个方向看会漏掉另一侧的抬高要求。

**复杂度。** 分时间、空间、契约三块看：

- 时间：两个函数都把数组扫常数遍——加油站一遍、糖果两遍（外加一次建 left 数组），都是 O(n)。
- 空间：加油站只用三个变量，是 O(1)；糖果用一个 left 数组，是 O(n)（right 已经用滚动变量压成 O(1) 了）。
- 具体感受：n=10⁵ 个加油站或孩子，也就是二三十万次加减法和比较，毫秒级完成。
- 契约：相邻同分不要求相同糖果数（题目只约束"更高的拿更多"）；空评分数组时两遍循环都不执行，返回总量 0。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的测试分两层。

**第一层：三条固定断言。**

- `assert can_complete_circuit([1,2,3,4,5],[3,4,5,1,2])==3`：正常值测试，就是正文手算的例子，锁定"前半程连亏三段、候选一路右移到 3"的答案。
- `assert can_complete_circuit([1],[2])==-1`：最小规模的不可达测试——只有一个站，加 1 升要花 2 升，总和 −1 < 0，应当返回 -1。它盯防的是"忘了看 total 就返回 start"的实现。
- `assert candy([1,0,2])==5`：正常值测试。手算：左遍 ratings 0<2 得 left=[1,1,2]；右遍 1>0 得右侧下界 [2,1,1]；逐点 max=[2,1,2]，总数 5。

**第二层：两组穷举对拍（小规模全枚举，不用随机）。**

- 糖果组：`product(range(3),repeat=4)` 枚举所有 3⁴=81 种长度 4、评分 0~2 的数组；对每个评分数组，再用 `product(range(1,5),repeat=4)` 枚举所有 4⁴=256 种每人 1~4 颗的分配，筛出满足全部相邻约束的（可行解集合），取其中总和最小值，和 `candy(ratings)` 对比。注意枚举里那个条件表达式：评分相等时不施加任何约束，正好对应题意。暴力直接按定义搜索"满足约束的最小总量"，与策略无关，是干净的标尺。
- 加油站组：`product(range(-2,3),repeat=4)` 枚举 5⁴=625 种差值序列（每位取 −2~2），再拆回 `gas=max(x,0)`、`cost=max(-x,0)`（一个非负、一个非负、差值还原）。对每个差值序列，暴力检查每个起点 s：从 s 出发把一圈每个"前 k 步累计"都算一遍，全非负才算合法起点，得到合法起点集合 `starts`。断言写的是 `(got in starts) if starts else got==-1`：如果存在合法起点，算法返回的下标必须是其中之一（注意契约允许返回任何一个合法起点，不要求唯一）；如果不存在，必须返回 -1。

In [4]:
assert can_complete_circuit([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]) == 3

assert can_complete_circuit([1], [2]) == -1

assert candy([1, 0, 2]) == 5

from itertools import product

for ratings in product(range(3), repeat=4):
    feasible = []
    for values in product(range(1, 5), repeat=4):
        if all((ratings[i] == ratings[i + 1] or (values[i] > values[i + 1] if ratings[i] > ratings[i + 1] else values[i] < values[i + 1]) for i in range(3))):
            feasible.append(sum(values))
    assert candy(ratings) == min(feasible)

for differences in product(range(-2, 3), repeat=4):
    gas = [max(x, 0) for x in differences]
    cost = [max(-x, 0) for x in differences]
    starts = [s for s in range(4) if all((sum((differences[(s + j) % 4] for j in range(k + 1))) >= 0 for k in range(4)))]
    got = can_complete_circuit(gas, cost)
    assert got in starts if starts else got == -1

print('N095: 所有本章断言通过')

N095: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 证明一次失败能排除一段起点。

**练习 2：** 为什么糖果只从左向右扫描不够。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1（证明一次失败排除一段起点）**：思路是把第一节的排除引理补完整。提示方向：设从 s 出发在 i 断油，即 sum(差值[s..i]) < 0；对 s 和 i 之间任意 s'，把这段差值拆成 [s..s'−1] 和 [s'..i] 两截。想想从 s 扫到 i 之前油箱从未断过这一事实，能推出第一截的和有什么符号？再由此推出第二截 sum(差值[s'..i]) 的符号必然也是负——这就是"从 s' 出发同样到不了 i+1"。写清楚输入输出后，建议手算一个差值 [3,−4,2] 的例子：从 0 出发在 1 断油，验证从 1 出发（和 2−4=−2）同样立刻断。最后可以用"枚举所有起点各跑一圈"的暴力小程序对拍本章接口。

**练习 2（为什么糖果只从左向右扫一遍不够）**：思路是找一条"右边还有下坡"的评分序列。提示方向：最短的反例只要 3 个位置，比如评分 [2,2,1]——左遍得到 [1,1,1]，看似满足"比左邻高多拿"（没有递增），但位置 1（评分 2）比右邻（评分 1）高，按题意必须拿得比右邻多，1 颗对 1 颗违规；补上右遍后右下界把位置 1 抬到 2，正确答案 [1,2,1] 共 4 颗。你要说明：左遍只保证了一半约束，右半约束必须靠反向扫描补齐，且合并必须逐点取最大（想一想：取平均或取左值分别会违反哪一侧）。验证可以用 “自动测试”部分 那种小规模穷举暴力（枚举每人的糖果数、筛可行、取最小）当裁判。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def can_complete_circuit(gas, cost):
    if len(gas) != len(cost) or not gas:
        return -1
    total = tank = start = 0
    for i, (g, c) in enumerate(zip(gas, cost)):
        total += g - c
        tank += g - c
        if tank < 0:
            start = i + 1
            tank = 0
    return start if total >= 0 else -1

def candy(ratings):
    n = len(ratings)
    left = [1] * n
    right = 1
    total = 0
    for i in range(1, n):
        if ratings[i] > ratings[i - 1]:
            left[i] = left[i - 1] + 1
    for i in range(n - 1, -1, -1):
        right = right + 1 if i < n - 1 and ratings[i] > ratings[i + 1] else 1
        total += max(left[i], right)
    return total

# 示例与回归测试
assert can_complete_circuit([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]) == 3

assert can_complete_circuit([1], [2]) == -1

assert candy([1, 0, 2]) == 5

from itertools import product

for ratings in product(range(3), repeat=4):
    feasible = []
    for values in product(range(1, 5), repeat=4):
        if all((ratings[i] == ratings[i + 1] or (values[i] > values[i + 1] if ratings[i] > ratings[i + 1] else values[i] < values[i + 1]) for i in range(3))):
            feasible.append(sum(values))
    assert candy(ratings) == min(feasible)

for differences in product(range(-2, 3), repeat=4):
    gas = [max(x, 0) for x in differences]
    cost = [max(-x, 0) for x in differences]
    starts = [s for s in range(4) if all((sum((differences[(s + j) % 4] for j in range(k + 1))) >= 0 for k in range(4)))]
    got = can_complete_circuit(gas, cost)
    assert got in starts if starts else got == -1

print('N095: 所有本章断言通过')

N095: 所有本章断言通过


## 11. 代表题与迁移

- **134. Gas Station（canonical）**：这题就是 `can_complete_circuit` 的原题——环形路线找可行起点。它练的是本章"前缀亏损 + 候选整段排除"的一遍扫描贪心。
- **135. Candy（canonical）**：这题就是 `candy` 的原题——双向相邻约束下的最小分配。它练的是"两遍扫描 + 逐点取最大合并下界"的套路，这是很多"左右各有一组不等式"题目的通用解法骨架。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。